In [10]:
!pip install -q streamlit
!npm install -q localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# --- 1. App Header & Setup ---
st.title("⚡ Electricity Demand Forecaster")
st.markdown("Predict hourly and daily electricity demand for the upcoming week.")

# --- 2. Load Assets from Google Drive ---
@st.cache_resource
def load_assets():
    # Use the Google Drive paths
    app_folder = '/content/drive/MyDrive/Mini Projects/Electricity Demand Forecasting/Electricity_App'
    model = joblib.load(f'{app_folder}/electricity_model.pkl')
    columns = joblib.load(f'{app_folder}/model_columns.pkl')
    historical_data = pd.read_csv(f'{app_folder}/last_historical_data.csv')
    return model, columns, historical_data

loaded_model, model_columns, historical_data = load_assets()

# --- 3. Sidebar for User Inputs ---
st.sidebar.header("Prediction Settings")
days_to_predict = st.sidebar.slider("Days to Forecast", min_value=1, max_value=14, value=7)
start_date = st.sidebar.date_input("Start Date", pd.Timestamp('2025-01-01'))

if st.sidebar.button("Generate Forecast"):
    with st.spinner('Calculating predictions...'):

        # --- 4. Prediction Logic ---
        future_start_date = pd.Timestamp(start_date)
        future_end_date = future_start_date + pd.Timedelta(days=days_to_predict) - pd.Timedelta(hours=1)
        future_dates = pd.date_range(start=future_start_date, end=future_end_date, freq='H')

        future_df = pd.DataFrame(index=future_dates)

        future_df['hour'] = future_df.index.hour
        future_df['dayofweek'] = future_df.index.dayofweek
        future_df['month'] = future_df.index.month
        future_df['year'] = future_df.index.year
        future_df['dayofyear'] = future_df.index.dayofyear
        future_df['weekofyear'] = future_df.index.isocalendar().week.astype(int)
        future_df['quarter'] = future_df.index.quarter
        future_df['is_weekend'] = future_df.index.dayofweek.isin([5, 6]).astype(int)

        future_df['Temperature'] = historical_data['Temperature'].iloc[-1]
        future_df['Humidity'] = historical_data['Humidity'].iloc[-1]
        future_df['Demand_lag_24hr'] = historical_data['Demand_lag_24hr'].iloc[-1]
        future_df['demand_lag_168hr'] = historical_data['demand_lag_168hr'].iloc[-1]
        future_df['demand_rolling_mean_24hr'] = historical_data['demand_rolling_mean_24hr'].iloc[-1]
        future_df['demand_rolling_std_24hr'] = historical_data['demand_rolling_std_24hr'].iloc[-1]

        for col in model_columns:
            if col not in future_df.columns:
                future_df[col] = 0

        future_df = future_df[model_columns]
        future_predictions_hourly = loaded_model.predict(future_df)
        future_df['Predicted Demand (Hourly)'] = future_predictions_hourly

        daily_predictions = future_df['Predicted Demand (Hourly)'].resample('D').sum()
        daily_prediction_df = pd.DataFrame({
            'Date': daily_predictions.index.date,
            'Predicted Daily Demand': daily_predictions.values
        })

        # --- 5. Visualizations ---
        st.success("Forecast Generated Successfully!")
        st.subheader("📊 Hourly Demand Forecast")
        st.line_chart(future_df['Predicted Demand (Hourly)'])
        st.subheader("📅 Daily Aggregated Demand")
        st.bar_chart(daily_prediction_df.set_index('Date')['Predicted Daily Demand'])

In [12]:
import urllib
print("Copy this IP Address / Password:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

Copy this IP Address / Password:
34.132.153.31


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

your url is: https://fair-cameras-cross.loca.lt
2026-06-14 11:23:01.326 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.132.153.31:8501

/content/app.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  future_dates = pd.date_range(start=future_start_date, end=future_end_date, freq='H')
/content/app.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  future_dates = pd.date_range(start=future_start_date, end=future_end_date, freq='H')
/content/app.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  future_dates = pd.date_range(start=future_start_date, end=future_end_date, freq='H')
/content/app.py:33: FutureWarning: 'H' is deprecated and will be removed in a future version, ple